In [ ]:
РАСЧИТЫВАЕМ ПРИЗНАКИ ДЛЯ DUBIA-текстов по аналогии с CHUNKING, 
чтобы использовать в моделях обученных на chunking-корпусах A/B/C

In [ ]:
===========

СКРИПТЫ генерации матриц признаков

0. Загрузка и аудит
1. MFW100, MFW300, MFW500, MFW1000
2. CHAR_3GRAM, CHAR_4GRAM
3. POS_18, POS_18_full, POS_2GRAM, POS_3GRAM
4. L3000, L3000_full, L1000, L1000_full
5. CHANKING - уточнить ID
    
===========

In [1]:
# = 0 =

# ЗАГРУЖАЕМ ФАЙЛ С ТЕКСТАМИ КОРПУСА И МЕТАДАННЫМИ

import pandas as pd

path = "/Users/anastasiabogdanova/R_directory/iskra-project/data/texts_with_metadata_20260401.csv"

df = pd.read_csv(path)

# print(df.head())
print(df.columns)


Index(['path', 'author_folder', 'file_name', 'text', 'n_chars', 'n_words'], dtype='object')


In [3]:
# = 0 =

# АУДИТ КОРПУСА 1

audit_authors = (
    df
    .groupby("author_folder")
    .agg(
        n_texts=("file_name", "count"),
        total_words=("n_words", "sum"),
        mean_words=("n_words", "mean"),
        median_words=("n_words", "median")
    )
    .sort_values(by="n_texts", ascending=False)
)

audit_authors.head(10)

,n_texts,total_words,mean_words,median_words
author_folder,,,,
lenin,46,212217,4613.413043,1743.5
martov,24,79755,3323.125000,2611.0
plehanov,24,94566,3940.250000,2192.5
parvus,20,80903,4045.150000,2360.5
trotsky,14,82352,5882.285714,2433.0
dubia,9,12166,1351.777778,1485.0
zasulich,8,40208,5026.000000,2655.5
krupskaya,6,18136,3022.666667,2104.0
ortodox,3,7543,2514.333333,2229.0


In [5]:
# = 0 =

# АУДИТ КОРПУСА 2

df.groupby("author_folder")["n_words"].describe()

,count,mean,std,min,25%,50%,75%,max
author_folder,,,,,,,,
dubia,9.0,1351.777778,651.218815,201.0,989.00,1485.0,1668.00,2120.0
krupskaya,6.0,3022.666667,2948.942703,776.0,1421.75,2104.0,2881.50,8817.0
lenin,46.0,4613.413043,11268.820346,435.0,1288.75,1743.5,2313.25,63216.0
martov,24.0,3323.125000,3193.721767,787.0,1698.25,2611.0,3101.25,15510.0
ortodox,3.0,2514.333333,696.320568,2006.0,2117.50,2229.0,2768.50,3308.0
parvus,20.0,4045.150000,7224.768387,404.0,1157.00,2360.5,3267.75,33529.0
plehanov,24.0,3940.250000,4648.401317,535.0,1213.00,2192.5,3202.25,18516.0
trotsky,14.0,5882.285714,9788.006943,487.0,1115.50,2433.0,4758.50,36298.0
zasulich,8.0,5026.000000,5820.788312,646.0,2101.75,2655.5,5114.75,18117.0


In [9]:
# = 0 =
# АУДИТ КОРПУСА 3. ВЫДЕЛЯЕМ ТОЛЬКО DUBIA-ТЕКСТЫ

# Оставляем только dubia
df_dubia_only = df[df["author_folder"] == "dubia"].copy()

print(df_dubia_only["author_folder"].value_counts())
print("Всего dubia-текстов:", len(df_dubia_only))
print("Было всего текстов:", len(df))

author_folder
dubia    9
Name: count, dtype: int64
Всего dubia-текстов: 9
Было всего текстов: 154


In [11]:
# = 0 =

# АУДИТ КОРПУСА 4. Без отфильтрованных текстов

df_dubia_only.groupby("author_folder")["n_words"].describe()

,count,mean,std,min,25%,50%,75%,max
author_folder,,,,,,,,
dubia,9.0,1351.777778,651.218815,201.0,989.0,1485.0,1668.0,2120.0


In [13]:
# = 0 =

# АУДИТ КОРПУСА 5.

df_dubia_only [["author_folder", "file_name", "n_words"]]

,author_folder,file_name,n_words
0,dubia,dubia_finans_manifest.txt,584
1,dubia,dubia_nasushnie_zadachi_I.txt,1485
2,dubia,dubia_novoe_poboishe_I.txt,1564
3,dubia,dubia_ot_red_iskry_I.txt,1437
4,dubia,dubia_ot_red_na_pismo_parvusa_I.txt,201
5,dubia,dubia_poslednee_slovo_bund_I.txt,989
6,dubia,dubia_priznaki_bankrotstva_I.txt,1668
7,dubia,dubia_slovo_mosc_vedomostyam_I.txt,2120
8,dubia,dubia_zakon_o_voznagr_I.txt,2118


In [ ]:
===========

1. MFW100, MFW300, MFW500, MFW1000
    
===========

In [15]:
# = 1 =

# 1.1 CHUNKING UPD

import re
import pandas as pd

chunk_size = 1000

def split_into_chunks(text, chunk_size=1000):
    """
    Разбивает текст на чанки по chunk_size слов.
    СОХРАНЯЕТ пунктуацию и регистр.
    """
    words = str(text).split()  # ← главное изменение: split() сохраняет всё!
    
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk_words = words[i:i + chunk_size]
        chunk_text = " ".join(chunk_words)
        chunks.append(chunk_text)
    
    return chunks

# Применяем чанкинг к исходному тексту
df_dubia_only["chunks_raw"] = df_dubia_only["text"].apply(
    lambda x: split_into_chunks(x, chunk_size=chunk_size)
)

# Разворачиваем в отдельные строки
new_rows = []

for _, row in df_dubia_only.iterrows():
    chunks = row["chunks_raw"]
    n_chunks = len(chunks)
    
    for i, chunk_text in enumerate(chunks):
        # Подсчет слов для метки is_short (только для статистики, text_raw не меняем)
        chunk_words = chunk_text.split()  # теперь split() возвращает слова с пунктуацией
        n_tokens = len(chunk_words)
        
        new_rows.append({
            "author": row["author_folder"],
            "file_name": row["file_name"],
            "chunk_id": f"{row['file_name']}_chunk{i+1}",
            "text_raw": chunk_text,        # ← теперь с пунктуацией!
            "n_words": n_tokens,
            "is_short": n_tokens < chunk_size
        })

df_chunks = pd.DataFrame(new_rows)

print(f"Количество чанков: {len(df_chunks)}")
print(f"Из них коротких (<{chunk_size} слов): {df_chunks['is_short'].sum()}")
df_chunks.head(10)

Количество чанков: 17
Из них коротких (<1000 слов): 9


,author,file_name,chunk_id,text_raw,n_words,is_short
0,dubia,dubia_finans_manifest.txt,dubia_finans_manifest.txt_chunk1,Финансовый манифест. Правительство на краю бан...,584,True
1,dubia,dubia_nasushnie_zadachi_I.txt,dubia_nasushnie_zadachi_I.txt_chunk1,Насущные задачи нашего движения. Русская социа...,1000,False
2,dubia,dubia_nasushnie_zadachi_I.txt,dubia_nasushnie_zadachi_I.txt_chunk2,"кружки, организуйтесь также и в политическую п...",485,True
3,dubia,dubia_novoe_poboishe_I.txt,dubia_novoe_poboishe_I.txt_chunk1,"НОВОЕ ПОБОИЩЕ. Повидимому, мы переживаем момен...",1000,False
4,dubia,dubia_novoe_poboishe_I.txt,dubia_novoe_poboishe_I.txt_chunk2,"Иванов, помощник начальника завода, т. е. то с...",564,True
5,dubia,dubia_ot_red_iskry_I.txt,dubia_ot_red_iskry_I.txt_chunk1,ЗАЯВЛЕНИЕ РЕДАКЦИИ «ИСКРЫ». ОТ РЕДАКЦИИ. Предп...,1000,False
6,dubia,dubia_ot_red_iskry_I.txt,dubia_ot_red_iskry_I.txt_chunk2,"в такую моду с легкой руки Эд. Бернштейна, П. ...",437,True
7,dubia,dubia_ot_red_na_pismo_parvusa_I.txt,dubia_ot_red_na_pismo_parvusa_I.txt_chunk1,От редакции. Печатая интересное письмо тов. Па...,201,True
8,dubia,dubia_poslednee_slovo_bund_I.txt,dubia_poslednee_slovo_bund_I.txt_chunk1,ПОСЛЕДНЕЕ СЛОВО БУНДОВСКОГО НАЦИОНАЛИЗМА. Загр...,989,True
9,dubia,dubia_priznaki_bankrotstva_I.txt,dubia_priznaki_bankrotstva_I.txt_chunk1,ПРИЗНАКИ БАНКРОТСТВА. Всего только год прошел ...,1000,False


In [17]:
# = 1 =

# 1.2 Проверка количества "хвостов" у каждого из авторов. Выделяем "усеченные" чанки

df_chunks.groupby("author")["is_short"].sum()

author
dubia    9
Name: is_short, dtype: int64

In [19]:
# = 1 =

# 1.3 Общая статистика полных и "коротких (<1000) чанков

df_chunks["is_short"].value_counts()

is_short
True     9
False    8
Name: count, dtype: int64

In [21]:
# = 1 =

# 1.4 Усечённые чанки по авторам

truncated_by_author = (
    df_chunks
    .groupby("author")["is_short"]
    .agg(
        is_short="sum",
        n_total="count",
        share_truncated="mean"
    )
    .sort_values("is_short", ascending=False)
)

truncated_by_author

,is_short,n_total,share_truncated
author,,,
dubia,9,17,0.529412


In [25]:
# = 1 =

# 1.5 Посмотреть на названия короких чанков

df_chunks[df_chunks["is_short"]][["author", "chunk_id", "n_words"]]

,author,chunk_id,n_words
0,dubia,dubia_finans_manifest.txt_chunk1,584
2,dubia,dubia_nasushnie_zadachi_I.txt_chunk2,485
4,dubia,dubia_novoe_poboishe_I.txt_chunk2,564
6,dubia,dubia_ot_red_iskry_I.txt_chunk2,437
7,dubia,dubia_ot_red_na_pismo_parvusa_I.txt_chunk1,201
8,dubia,dubia_poslednee_slovo_bund_I.txt_chunk1,989
10,dubia,dubia_priznaki_bankrotstva_I.txt_chunk2,668
13,dubia,dubia_slovo_mosc_vedomostyam_I.txt_chunk3,120
16,dubia,dubia_zakon_o_voznagr_I.txt_chunk3,118


In [29]:
# = 1 =

# 1.6 Проверяем сколько у нас коротких чанков и чьих

# Чанки от 100 до 500 слов (не включая 500)
chunks_100_500 = df_chunks[(df_chunks["n_words"] >= 100) & (df_chunks["n_words"] < 500)]

print(f"=== ЧАНКИ ОТ 100 ДО 500 СЛОВ ===")
print(f"Всего таких чанков: {len(chunks_100_500)}")
print(f"Из них коротких (is_short=True): {chunks_100_500['is_short'].sum()}")
print()

# По авторам
print("=== РАСПРЕДЕЛЕНИЕ ПО АВТОРАМ ===")
author_dist = chunks_100_500.groupby("author").size().sort_values(ascending=False)
print(author_dist)
print()

# Детальный вывод с названиями и количеством слов
print("=== ДЕТАЛЬНЫЙ СПИСОК (первые 20) ===")
display(chunks_100_500[["author", "file_name", "chunk_id", "n_words"]].head(20))

# Статистика по авторам
print("\n=== СТАТИСТИКА ПО АВТОРАМ ===")
stats = chunks_100_500.groupby("author")["n_words"].agg(["count", "min", "max", "mean"]).round(1)
print(stats)

# Можно также посмотреть, сколько чанков теряет каждый автор при пороге 200
print("\n=== ПОТЕРИ ПРИ ПЕРЕХОДЕ К ПОРОГУ 200 СЛОВ ===")
before = df_chunks.groupby("author").size()
after_200 = df_chunks[df_chunks["n_words"] >= 200].groupby("author").size()
loss = before - after_200
loss_df = pd.DataFrame({
    "всего_чанков": before,
    "после_фильтра_200": after_200,
    "потеряно": loss,
    "потеря_%": (loss / before * 100).round(1)
})
print(loss_df)

=== ЧАНКИ ОТ 100 ДО 500 СЛОВ ===
Всего таких чанков: 5
Из них коротких (is_short=True): 5

=== РАСПРЕДЕЛЕНИЕ ПО АВТОРАМ ===
author
dubia    5
dtype: int64

=== ДЕТАЛЬНЫЙ СПИСОК (первые 20) ===


,author,file_name,chunk_id,n_words
2,dubia,dubia_nasushnie_zadachi_I.txt,dubia_nasushnie_zadachi_I.txt_chunk2,485
6,dubia,dubia_ot_red_iskry_I.txt,dubia_ot_red_iskry_I.txt_chunk2,437
7,dubia,dubia_ot_red_na_pismo_parvusa_I.txt,dubia_ot_red_na_pismo_parvusa_I.txt_chunk1,201
13,dubia,dubia_slovo_mosc_vedomostyam_I.txt,dubia_slovo_mosc_vedomostyam_I.txt_chunk3,120
16,dubia,dubia_zakon_o_voznagr_I.txt,dubia_zakon_o_voznagr_I.txt_chunk3,118



=== СТАТИСТИКА ПО АВТОРАМ ===
        count  min  max   mean
author                        
dubia       5  118  485  272.2

=== ПОТЕРИ ПРИ ПЕРЕХОДЕ К ПОРОГУ 200 СЛОВ ===
        всего_чанков  после_фильтра_200  потеряно  потеря_%
author                                                     
dubia             17                 15         2      11.8


In [39]:
# = 1 =

# = 1.7

def create_dubia_corpus_with_filter(df_chunks, min_words=200):
    """
    Создает корпус из чанков dubia.
    Удаляет только чанки короче min_words.
    """
    # Фильтруем по длине
    corpus_dubia = df_chunks[df_chunks["n_words"] >= min_words].copy()
    
    # Сортируем по chunk_id
    corpus_dubia = corpus_dubia.sort_values("chunk_id").reset_index(drop=True)
    
    return corpus_dubia


# Создаем корпус дубиа (мин. 200 слов)
corpus_dubia = create_dubia_corpus_with_filter(df_chunks, min_words=200)

print("=== КОРПУС DUBIA (очищенный от чанков <200 слов) ===")
print(f"Было чанков: {len(df_chunks)}")
print(f"Стало чанков: {len(corpus_dubia)}")
print(f"Удалено чанков: {len(df_chunks) - len(corpus_dubia)}")

print("\nРаспределение по авторам (все dubia):")
print(corpus_dubia["author"].value_counts())

print("\nДлина оставшихся чанков:")
print(corpus_dubia["n_words"].describe())

print("\nЧанки, которые остались:")
print(corpus_dubia[["chunk_id", "n_words"]])

=== КОРПУС DUBIA (очищенный от чанков <200 слов) ===
Было чанков: 17
Стало чанков: 15
Удалено чанков: 2

Распределение по авторам (все dubia):
author
dubia    15
Name: count, dtype: int64

Длина оставшихся чанков:
count      15.000000
mean      795.200000
std       275.803553
min       201.000000
25%       574.000000
50%      1000.000000
75%      1000.000000
max      1000.000000
Name: n_words, dtype: float64

Чанки, которые остались:
                                      chunk_id  n_words
0             dubia_finans_manifest.txt_chunk1      584
1         dubia_nasushnie_zadachi_I.txt_chunk1     1000
2         dubia_nasushnie_zadachi_I.txt_chunk2      485
3            dubia_novoe_poboishe_I.txt_chunk1     1000
4            dubia_novoe_poboishe_I.txt_chunk2      564
5              dubia_ot_red_iskry_I.txt_chunk1     1000
6              dubia_ot_red_iskry_I.txt_chunk2      437
7   dubia_ot_red_na_pismo_parvusa_I.txt_chunk1      201
8      dubia_poslednee_slovo_bund_I.txt_chunk1      989
9 

In [43]:
# = 1 =

# = 1.8 СОХРАНЕНИЕ КОРПУСА DUBIA

import os

# Колонки для сохранения (те же, что и для основного корпуса)
cols = ["author", "file_name", "chunk_id", "text_raw", "n_words"]


# Функция для сохранения с сортировкой (адаптирована для дубиа)
def save_dubia_corpus(corpus, name):
    # Берем только те колонки, которые существуют в corpus
    available_cols = [col for col in cols if col in corpus.columns]
    df_sorted = corpus[available_cols].copy()
    df_sorted = df_sorted.sort_values(["author", "file_name", "chunk_id"])
    df_sorted.to_csv(name, index=False, encoding="utf-8")
    print(f"Сохранен: {name} ({len(df_sorted)} строк)")

# Путь к папке (такой же, как для основного корпуса)
output_dir = "/Users/anastasiabogdanova/R_directory/iskra-project/data/processed/"

# Убедимся, что папка существует
os.makedirs(output_dir, exist_ok=True)

# Сохраняем корпус дубиа (если ты создала corpus_dubia)
# Вариант 1: если у тебя есть corpus_dubia (после фильтрации)
if 'corpus_dubia' in locals():
    save_dubia_corpus(corpus_dubia, os.path.join(output_dir, "corpus_dubia.csv"))

# Вариант 2: если ты хочешь сохранить原始 df_chunks (без фильтрации)
# save_dubia_corpus(df_chunks, os.path.join(output_dir, "corpus_dubia_raw.csv"))

print("\nГотово! Корпус дубиа сохранен в /data/processed/")

Сохранен: /Users/anastasiabogdanova/R_directory/iskra-project/data/processed/corpus_dubia.csv (15 строк)

Готово! Корпус дубиа сохранен в /data/processed/


In [45]:
# = 1 =

# 1.9 Проверка: есть ли пунктуация в text_raw?

sample_text = df_chunks.iloc[0]["text_raw"]
print(sample_text[:200])
print("\nСодержит точку?" , "." in sample_text)
print("Содержит запятую?" , "," in sample_text)
print("Содержит тире?" , "-" in sample_text or "—" in sample_text)

Финансовый манифест. Правительство на краю банкротства. Оно превратило страну в развалины и усеяло их трупами. Измученные и изголодавшиеся крестьяне не в состоянии платить подати. Правительство на нар

Содержит точку? True
Содержит запятую? True
Содержит тире? True


In [ ]:
# = 1 =

In [ ]:
# = 1 =

In [ ]:
# = 1 =

In [ ]:
# = 1 =

In [ ]:
Многоклассовая классификация

- Logistic Regression (Логистическая регрессия)
- SVM (Опорные векторы)
- Random Forest (Случайный лес)

In [ ]:
Деревья решений и правил
Чтобы повысить точность и стабильность классификации, используют ансамблевые методы: бэггинг, случайный лес и бустинг.
Бэггинг (сокр. от Bootstrap Aggregating) — это способ собрать «консилиум» из моделей.
Работает он так:
Случайные группы (бутстрэп): мы берем наш исходный список данных и много раз вытягиваем из него случайные подмножества. 
Важно: мы выбираем именно наблюдения (строки). Один и тот же текст может попасть в одну подвыборку несколько раз, а в другую — ни разу.
Обучение: на каждом таком случайном наборе мы учим отдельное дерево.
Голосование (агрегация): когда нужно классифицировать новый объект, мы спрашиваем каждое дерево: “Это какой класс?”. 
В итоге побеждает тот вариант, за который проголосовало большинство.

Random Forest (Случайный лес) — это бэггинг «в квадрате».
К случайному выбору наблюдений (строк) добавляется случайный выбор признаков (колонок). 
Это заставляет деревья искать разные закономерности и не дает им всем совершать одну и ту же ошибку. 
Например, если в данных есть один супер-сильный признак, обычный бэггинг построит все деревья вокруг него, и они будут одинаковыми. 
Случайный выбор колонок (признаков) заставляет модель смотреть на данные под разными углами.

Бустинг — это «работа над ошибками». Градиентные бустинговые деревья
Здесь деревья строятся по очереди. Первое дерево пытается классифицировать данные как может. 
Второе дерево внимательно смотрит на те наблюдения (строки), где первое дерево ошиблось, и пытается исправить именно их. 
Третье исправляет ошибки первых двух. Так модель постепенно «вытягивает» самые сложные случаи.

(LDA) линейный дискриминантный анализ 
Regularized Discriminant Analysis (RDA) - компромисс между линейным (LDA) и квадратичным дискриминантным анализом (QDA) 
с добавлением регуляризации.
Метод ближайших соседей (K-Nearest Neighbors — KNN) - классификация (или регрессия) объекта производится на основе меток 
(или значений) K ближайших к нему объектов из обучающей выборки.